# 🍽️ Dining Insights During Global Events: Event-Based Dining Trend Analysis
### MR603 Thesis — Multimodal AI Framework

---
### 📋 Notebook Overview
This notebook implements a **multimodal analysis pipeline** for dining trend analysis during global events (Olympics, FIFA World Cup). It covers:
1. Environment setup (CERN LCG / cvmfs compatible)
2. Dataset loading & exploration (Food-101, Tokyo Olympics Tweets, FIFA Tweets, Yelp Reviews)
3. Text preprocessing & NLP (BERT-based sentiment + BERTopic)
4. Image preprocessing & classification (ResNet-50 / ViT)
5. Multimodal fusion
6. Evaluation & visualisation
7. Interpretability (SHAP + Grad-CAM)

**Datasets used:**
- 🐦 [Tokyo Olympics 2020 Tweets](https://www.kaggle.com/datasets/gpreda/tokyo-olympics-2020-tweets)
- 🐦 [FIFA World Cup 2022 Tweets](https://www.kaggle.com/datasets/tirendazacademy/fifa-world-cup-2022-tweets)
- 🍕 [Food-101 Images](https://huggingface.co/datasets/ethz/food101)
- ⭐ [Yelp Reviews](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset)

> **Note:** This notebook is configured for **CERN LCG / cvmfs** environments with read-only system site-packages. All installs use `--user` flag.

---
## 📦 Section 1: Environment Setup (CERN LCG Compatible)

In [1]:
import sys
import os
import site

# ── Make user-installed binaries & packages accessible ──────────────────
user_bin  = os.path.expanduser('~/.local/bin')
user_site = site.getusersitepackages()

if user_bin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = user_bin + ':' + os.environ.get('PATH', '')

if user_site not in sys.path:
    sys.path.insert(0, user_site)

print(f'✅ PATH includes: {user_bin}')
print(f'✅ sys.path includes: {user_site}')
print(f'✅ Python executable: {sys.executable}')

✅ PATH includes: /eos/user/r/rpaudel/.local/bin
✅ sys.path includes: /eos/user/r/rpaudel/.local/lib/python3.12/site-packages
✅ Python executable: /cvmfs/sft.cern.ch/lcg/views/LCG_108_cuda/x86_64-el9-gcc13-opt/bin/python3


In [2]:
# ── Install all required packages to user directory ──────────────────────
# Using --user flag because /cvmfs is read-only on CERN systems

packages = [
    'kaggle',
    'transformers',
    'datasets',
    'scikit-learn',
    'bertopic',
    'umap-learn',
    'hdbscan',
    'shap',
    'grad-cam',
    'wordcloud',
    'plotly',
    'kaleido',
    'tqdm'
]

for pkg in packages:
    ret = os.system(f'{sys.executable} -m pip install --user {pkg} -q')
    status = '✅' if ret == 0 else '⚠️'
    print(f'{status} {pkg}')

# Reload sys.path so newly installed packages are importable immediately
import importlib
import site as _site
importlib.reload(_site)
user_site = _site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

print('\n✅ All installations complete.')

✅ kaggle
✅ transformers
✅ datasets
✅ scikit-learn
✅ bertopic
✅ umap-learn
✅ hdbscan
✅ shap
✅ grad-cam
✅ wordcloud
✅ plotly
✅ kaleido
✅ tqdm

✅ All installations complete.


In [3]:
# ── Core imports ─────────────────────────────────────────────────────────
import json
import re
import glob
import gzip
import pathlib
import warnings
import unicodedata
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from wordcloud import WordCloud
import plotly.express as px

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image

# HuggingFace
from datasets import load_dataset
from transformers import (
    BertTokenizer, BertModel, BertForSequenceClassification,
    ViTForImageClassification, ViTImageProcessor,
    pipeline, TrainingArguments, Trainer
)

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, mean_absolute_error
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device      : {device}')
print(f'✅ PyTorch     : {torch.__version__}')
if torch.cuda.is_available():
    print(f'✅ GPU         : {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Output directory
os.makedirs('./outputs', exist_ok=True)
os.makedirs('./models',  exist_ok=True)
os.makedirs('./data/olympics', exist_ok=True)
os.makedirs('./data/fifa',     exist_ok=True)
os.makedirs('./data/yelp',     exist_ok=True)
print('✅ Output dirs : ./outputs/, ./models/, ./data/')

Matplotlib is building the font cache; this may take a moment.
2026-03-01 12:47:07.653179: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772365627.671509     634 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772365627.677408     634 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772365627.694484     634 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772365627.694520     634 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772365627.6

✅ Device      : cuda
✅ PyTorch     : 2.9.1+cu128
✅ GPU         : NVIDIA A100-PCIE-40GB MIG 2g.10gb
✅ VRAM        : 10.5 GB
✅ Output dirs : ./outputs/, ./models/, ./data/


---
## 📂 Section 2: Dataset Loading & Exploration

### 2.1 Kaggle API Setup
> Paste your Kaggle credentials below. Get them from https://www.kaggle.com/settings → API → Create New Token

In [4]:
# ── Kaggle credentials ───────────────────────────────────────────────────
# Replace these with your actual Kaggle username and API key
KAGGLE_USERNAME = 'rishikesh0523'   # ← replace
KAGGLE_KEY      = 'KGAT_1a249a6d6e253612007e0eb54e0df8b7'    # ← replace

# Write kaggle.json to ~/.kaggle/
kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
creds_path = kaggle_dir / 'kaggle.json'
creds_path.write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
creds_path.chmod(0o600)

# Authenticate using Kaggle Python API (no CLI needed)
import kaggle
kaggle.api.authenticate()
print('✅ Kaggle authenticated successfully')

✅ Kaggle authenticated successfully


### 2.2 Download All Kaggle Datasets

In [5]:
# ── Download Tokyo Olympics 2020 Tweets ──────────────────────────────────
print('Downloading Tokyo Olympics 2020 Tweets...')
kaggle.api.dataset_download_files(
    'gpreda/tokyo-olympics-2020-tweets',
    path='./data/olympics',
    unzip=True,
    quiet=False
)
print('✅ Olympics tweets downloaded')

# ── Download FIFA World Cup 2022 Tweets ──────────────────────────────────
print('\nDownloading FIFA World Cup 2022 Tweets...')
kaggle.api.dataset_download_files(
    'tirendazacademy/fifa-world-cup-2022-tweets',
    path='./data/fifa',
    unzip=True,
    quiet=False
)
print('✅ FIFA tweets downloaded')

# ── Download Yelp Reviews (only the review JSON — rest is very large) ────
print('\nDownloading Yelp reviews JSON...')
try:
    kaggle.api.dataset_download_file(
        'yelp-dataset/yelp-dataset',
        file_name='yelp_academic_dataset_review.json',
        path='./data/yelp',
        quiet=False
    )
    print('✅ Yelp reviews downloaded')
except Exception as e:
    print(f'⚠️  Yelp single-file download failed ({e}). Trying full dataset...')
    kaggle.api.dataset_download_files(
        'yelp-dataset/yelp-dataset',
        path='./data/yelp',
        unzip=True,
        quiet=False
    )
    print('✅ Yelp full dataset downloaded')

# ── Show downloaded files ─────────────────────────────────────────────────
print('\n── Downloaded files ──')
for folder in ['./data/olympics', './data/fifa', './data/yelp']:
    files = glob.glob(f'{folder}/**/*', recursive=True)
    print(f'  {folder}:')
    for f in files:
        size = os.path.getsize(f) / 1e6 if os.path.isfile(f) else 0
        print(f'    {f}  ({size:.1f} MB)')

Dataset URL: https://www.kaggle.com/datasets/gpreda/tokyo-olympics-2020-tweets


100%|██████████| 24.8M/24.8M [00:01<00:00, 14.3MB/s]



✅ Olympics tweets downloaded

Dataset URL: https://www.kaggle.com/datasets/tirendazacademy/fifa-world-cup-2022-tweets


100%|██████████| 1.38M/1.38M [00:00<00:00, 2.36MB/s]



✅ FIFA tweets downloaded

Dataset URL: https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset


100%|██████████| 2.07G/2.07G [00:59<00:00, 37.1MB/s]



✅ Yelp reviews downloaded

── Downloaded files ──
  ./data/olympics:
    ./data/olympics/tokyo_2020_tweets.csv  (62.8 MB)
  ./data/fifa:
    ./data/fifa/fifa_world_cup_2022_tweets.csv  (4.5 MB)
  ./data/yelp:
    ./data/yelp/yelp_academic_dataset_review.json.zip  (2220.6 MB)
